# First Title  
## Second Title  
- item 1
- item 2
- item 3
**abcde**

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [10]:
df=pd.read_csv("RFM_Raw.csv")
df.head()

,Date,InvoNo,CustomerKey,Amount
0,2018/1/1,JA1803995,A0620,411.400
1,2018/1/1,JA1805774,A1336,658.240
2,2018/1/1,JA1801197,A0786,1892.352
3,2018/1/2,JA1805224,A0937,411.400
4,2018/1/2,JA1802973,A1371,411.400


In [11]:
import pandas as pd

# 假設你的原始資料長這樣：
# data = {
#     'CustomerKey': ['C001', 'C001', 'C002', 'C002', 'C002'],
#     'Date': ['2026-05-01', '2026-05-10', '2026-05-03', '2026-05-15', '2026-05-20'],
#     'InvoNo': ['INV001', 'INV002', 'INV003', 'INV004', 'INV005'],
#     'Amount': [100, 200, 150, 300, 50],
# }
# df = pd.DataFrame(data)

# 1. 確保「Date」欄位是 datetime 型態，以便後續做Date減法
df['Date'] = pd.to_datetime(df['Date'])

# 2. 找出整個資料集中的「最大Date」作為基準點
max_date = df['Date'].max()

# 3. 進行 groupby 並計算 F、M、R
rfm_df = (
    df.groupby('CustomerKey')
    .agg(
        # F: InvoNo計數（通常用 count 或 nunique，這裡依需求用 count）
        F=('InvoNo', 'count'),
        # M: Amount加總
        M=('Amount', 'sum'),
        # 先找出該客戶的最大交易Date，用來算 R
        max_transaction_date=('Date', 'max'),
    )
    .reset_index()
)

# 4. 計算 R：基準最大Date 與 該客戶最大交易Date 的差額
# .dt.days 可以將時間差轉換成整數的天數
rfm_df['R'] = (max_date - rfm_df['max_transaction_date']).dt.days

# 5. 調整欄位名稱與順序，符合你的要求：Customer, F, M, R
rfm_df = rfm_df.rename(columns={'CustomerKey': 'Customer'})

# 移除中途幫忙計算而暫存的 max_transaction_date 欄位，並只保留需要的四個欄位
rfm_df = rfm_df[['Customer', 'F', 'M', 'R']]

# 檢視結果
print(rfm_df)

    Customer   F            M   R
0      A0350  25  51238.11440   9
1      A0351   6  17890.31200  16
2      A0352  14  46424.03852  14
3      A0353  14  27607.68636  24
4      A0354   6   9973.92064  31
..       ...  ..          ...  ..
727    A1413  23  52738.79456  10
728    A1415  20  53019.49664  17
729    A1416  24  49938.38378   4
730    A1419  22  64257.84520  50
731    A1420  14  25054.18342  85

[732 rows x 4 columns]


In [14]:
rfm_df.to_csv("RFM_Data.csv",index = False)